# 04 · Signal QC

**What this notebook does.** It checks the signal quality of every channel in the 60 final sessions: the power spectrum,
the power in each frequency band, the 50 Hz mains line, and whether a channel is an **outlier compared with its neighbours**.
Channels found here (and in 05) are added to `extra_bad_channels.csv` and are removed in all analyses.

**Run order in the project:** 01 → 02 → 03 → **04** → 05 → 06 → 07. Needs `final_sessions.csv` and `extra_bad_channels.csv`.
The loops were run once; they do not need to be run again unless sessions or bad channels change.

**Steps**

| Step | What it does | Output |
|---|---|---|
| 1 | QC of every session, original and CMR (only with `RUN = True`) | per session `QC\psd.npz`, `QC\qc_channels.csv`; `qc_channels_all.csv` |
| 2 | Summary: outliers and 50 Hz per session and rat, repeat offenders | `qc_sessions.csv`, `qc_repeat_outliers.csv` |
| 3 | QC figures of one session | figures |
| 4 | QC of every session for the LOCAL reference + comparison of the 3 references | per session `QC\psd_local.npz`, `qc_channels_local.csv`; `qc_channels_all_local.csv` |
| 5 | Example: one bad channel next to its neighbours (slide figure) | figure |
| 6 | Fixed electrical interference in rat 135 | figure |

Removed from the old notebook: three test cells on single sessions (they repeated what step 1 does for all sessions) and a
one-off search for the most deviating channel (only used to choose the example of step 5).

**How to use it:** run the settings cell first; then any cell. Nothing is written unless `SAVE = True` (figures and summary tables)
or `RUN = True` (the per-session QC files). Functions: `fx_qc.py` (+ `functions.py`); colours and saving: `fx_common.py`.

**The three references** (made by `functions.load_session` when a session is loaded; the saved recording itself has no reference):
- **original**: the 1000 Hz recording as saved (no re-referencing);
- **CMR** (common median reference): every channel minus the median of all channels at each time point. Removes signal shared by the whole probe;
- **LOCAL**: every channel minus the median of the channels 30–200 µm away on the probe. Removes more local shared signal.

Before referencing, the channels removed in preprocessing are already missing and the session's channels from
`extra_bad_channels.csv` are removed, so bad channels can't leak into the reference.

## Settings

**Paths:** `CODE_DIR`, `DATA_ROOT` with `PROC` (sessions; the per-session QC files go into `PROC\<session>\amir_data\QC`), `AMIR`,
`TABLES` (temporarily the old folder), `FIG_DIR` (figures of this notebook), `QC_DIR` (slide figures of steps 5–6).

**Saving:** `SAVE = False` writes no figure or summary table; `True` writes figures to `FIG_DIR` and tables to `TABLES`.
To save one cell elsewhere, add `folder=r"...", force=True` to its save line.

**Loops:** `RUN = False` only reports which sessions are done. `True` computes the sessions without a finished QC file;
`OVERWRITE = True` redoes all of them. Time: about 1–2 min per session.

**QC rule** (inside `functions.py`): a channel is a **local outlier** if its broadband power differs from the median of its neighbours
(same shank, within ±75 µm, at least 3 neighbours) by more than 0.3 log10 units (a factor 2). The **50 Hz line ratio** = power at
49–51 Hz / power at 45–48 + 52–55 Hz; above 1 means the notch filter left a line.

**Examples:** `PLOT_SESSION` (step 3), `EX_SESSION`, `EX_CHANNEL` (step 5).

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

# ── paths ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py + functions.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings (+ per-session QC / ERP files)
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder (all outputs)
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
FIG_DIR   = os.path.join(AMIR, "figures", "04_QC")                                 # figures of this notebook (if SAVE)
QC_DIR    = os.path.join(AMIR, "QC")                                               # slide figures of steps 5–6 (if SAVE)

# ── saving and loops ──
SAVE      = False                                                                  # True = write figures and summary tables
RUN       = False                                                                  # True = compute missing per-session QC files
OVERWRITE = False                                                                  # True = recompute every session

# ── examples ──
PLOT_SESSION = "20260529_r0000136_wt_1318"                                         # step 3
EX_SESSION, EX_CHANNEL = "20260611_r0000135_wt_1415", "AP326"                      # step 5: one bad channel

# ── code + tables ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, functions as fn, fx_qc as fq
for m_ in (fxc, fn, fq): importlib.reload(m_)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
fxc.check_paths(PROC=PROC, TABLES=TABLES)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)
final = pd.read_csv(os.path.join(TABLES, "final_sessions.csv"), dtype={"rat": str})
XB    = pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv"))
REFS  = ["original", "CMR"]
print(f"final sessions: {len(final)} | extra bad channel rows: {len(XB)}")

## Step 1 · Signal QC of all final sessions (original and CMR)

**What it does, per session:**
1. loads the recording (minus the extra bad channels) in the original and CMR reference;
2. **power spectrum** of every channel (Welch): 30 pieces of 10 s spread over the whole recording, 2 s windows (0.5 Hz resolution), 1–100 Hz;
3. per channel: log10 power in each band (delta 1–3, theta 4–7, alpha 8–12, beta 13–29, low gamma 30–60, high gamma 61–100 Hz),
   broadband power, and the 50 Hz line ratio;
4. **local outlier test:** broadband power vs the median of the neighbours (±75 µm, same shank); flagged if |Δlog10| > 0.3.
   Why neighbours and not the whole probe: power changes with depth (different layers and regions), so only a single odd channel
   among similar neighbours points to a broken channel.

**With `RUN = True`** it writes per session `QC\psd.npz` (frequencies, spectra per reference, channel ids) and `QC\qc_channels.csv`
(written last = session finished). Then all finished sessions are collected into one table.

**Saves:** `qc_channels_all.csv` (one row per session × reference × channel).

In [ ]:
fq.run_signal_qc(final, PROC, XB, run=RUN, overwrite=OVERWRITE, err_log=os.path.join(TABLES, "qc_errors.csv"))
qc_all = fq.collect_qc(final, PROC)
if qc_all is not None: fxc.save_table(qc_all, "qc_channels_all")

## Step 2 · QC summary

**What it does.** From all channels of all sessions:
- per session and reference: number of channels, local outliers, channels with a 50 Hz ratio > 1, median 50 Hz ratio;
- per rat: sums and the session with the most outliers;
- **repeat offenders:** the same readout channel (AP number) or the same electrode position (x, y) flagged in at least 3 sessions
  **and** in at least 30 % of the sessions that recorded it. A channel that is odd again and again is likely broken, not a one-off.

**Input:** the collected QC table (from the per-session files). **Saves:** `qc_sessions.csv`, `qc_repeat_outliers.csv`.

In [ ]:
qc_sessions, qc_repeat = fq.qc_summary(fq.collect_qc(final, PROC), final)
fxc.save_table(qc_sessions, "qc_sessions"); fxc.save_table(qc_repeat, "qc_repeat_outliers")

## Step 3 · QC figures of one session

**What it does** for `PLOT_SESSION`, from its saved spectra (step 1):
- **power by depth**, per shank: top = log10 power (depth × frequency); bottom = the same minus the shank's median spectrum, so a single
  odd channel stands out as a stripe; original and CMR;
- the **median spectrum** over channels, original vs CMR;
- **4 s of raw LFP** from 20 channels spread over the probe (`functions.select_channels`), middle of the session, original and CMR.

**Saves (if `SAVE`):** the figures in `FIG_DIR\<session>`.

In [ ]:
z = np.load(fq.qc_files(PROC, PLOT_SESSION)[1], allow_pickle=True)
freqs, ids, psd = z["freqs"], [str(c) for c in z["channel_ids"]], {r: z[f"psd_{r}"] for r in REFS}
recs, _ = fn.load_session(PROC, PLOT_SESSION, XB); pdf = fn.probe_table(recs["original"]); sel = fn.select_channels(pdf, n_total=20)
out = os.path.join(FIG_DIR, PLOT_SESSION) if SAVE else None
for ref in REFS:
    fn.plot_psd_depth(freqs, psd[ref], ids, pdf, f"{PLOT_SESSION} — PSD by depth ({ref})", save_path=out and os.path.join(out, f"psd_depth_{ref.lower()}.png"))
fn.plot_mean_psd(psd, freqs, f"{PLOT_SESSION} — median PSD over channels", save_path=out and os.path.join(out, "psd_original_vs_cmr.png"))
for ref in REFS:
    fn.plot_traces(recs[ref], sel, recs["original"].get_total_duration() / 2, 4, f"{PLOT_SESSION} — raw LFP ({ref})",
                   save_path=out and os.path.join(out, f"traces_{ref.lower()}.png"))

## Step 4 · Signal QC for the LOCAL reference

**What it does.** The same QC as step 1, now on the **LOCAL** reference, which the burst and coupling results use.
With `RUN = True` it writes per session `QC\psd_local.npz` and `QC\qc_channels_local.csv` (the original and CMR files are not changed).

**Summary:** local outliers and 50 Hz per rat for original, CMR and LOCAL side by side, and the LOCAL repeat offenders (same rule as step 2).

**Saves:** `qc_channels_all_local.csv`.

In [ ]:
fq.run_signal_qc(final, PROC, XB, run=RUN, overwrite=OVERWRITE, local=True)
qc_local = fq.collect_qc(final, PROC, local=True)
if qc_local is not None:
    fxc.save_table(qc_local, "qc_channels_all_local")
    qc_local_repeat = fq.qc_local_summary(qc_local, fq.collect_qc(final, PROC))

## Step 5 · Example: a bad channel next to its neighbours

**What it does.** For one bad channel (`EX_CHANNEL` in `EX_SESSION`), loaded **without** the extra-bad list so the channel is still there
(original reference):
- **A** broadband power of every channel along the shank; grey band = neighbour median ±0.3 log10 (factor 2), the QC limit;
- **B** power spectrum of the channel (red) and its neighbours within ±75 µm (grey), same method as step 1;
- **C** 2 s of raw LFP of the channel and its 4 nearest neighbours.

It prints how far the channel's power is from its neighbours (log10 and as a factor).

**Saves:** `QC example bad channel.png` in `QC_DIR`.

In [ ]:
fig, dev = fq.example_bad_channel(PROC, EX_SESSION, EX_CHANNEL)
fxc.save_fig(fig, "QC example bad channel", folder=QC_DIR); plt.show()

## Step 6 · Fixed electrical interference in rat 135

**What it does.** Draws the median spectrum (over channels, original reference) of every session between 30 and 70 Hz, coloured by rat.
Peaks at about 42.5 and 57.5 Hz appear in every session of rat 135 only, i.e. a fixed electrical interference of that rat's setup.
This matters for the low-gamma band (30–60 Hz) in rat 135.

**Input:** the saved spectra of step 1. **Saves:** `fixed_electrical_interference_rat_135.png` in `QC_DIR`.

In [ ]:
fig = fq.interference_per_rat(final, PROC)
fxc.save_fig(fig, "fixed_electrical_interference_rat_135", folder=QC_DIR); plt.show()